# Calculate leverage in PIM portofolios for BNP

In [24]:
# libraries, libraries!
import time
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np
import re # to extract dates
from tqdm import tqdm
from constants import pthCmp
from utilities import timediff

In [25]:
# list the fund holdings files

start_time = time.time()
start_time_lvrg = time.time()
pthL = pthCmp + r"\BNP Leverage DD\Annual Review Data"
files = [
    f.name
    for f in Path(pthL).iterdir()
    if f.is_file()
    and f.name.startswith("Portfolio Analytics Report")
    and f.name.endswith(".xlsx")
]

# for file in files:
#     print(file)
print(f"{len(files)} files")

print(timediff(start_time, time.time()))

12 files
0.6sec


In [26]:
# list the funds

pthF = pthCmp + r"\BNP Leverage DD\Prescient additional information.xlsx"
df2 = pd.read_excel(pthF, usecols = "N")
funds = df2.iloc[:,0].unique()
print(len(funds), funds)

21 ['QIFFGIF' 'PCGEARF' 'PGCBF' 'PGCEF' 'PGPCEM_C' 'IPIPF' 'NFMWAGG' 'PEQ'
 'PPSBAL_C' 'PABS' 'PIMBAL' 'PSIF' 'MOMTAAHI' 'MOMTAALI' 'MOMTAAMI'
 'SAAMCAU' 'SAAMINC' 'SAAMMOD' 'BPROV' 'UCTRFBAL' 'UNISABAL']


In [27]:
# load each holdings file and then create a single dataframe from them
df_all = []
for file in tqdm(files):
    df_f = pd.read_excel(pthL + '\\' + file)
    df_all.append(df_f)

df = pd.concat(df_all, ignore_index = True)
# df

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [00:19<00:00,  1.59s/it]


In [28]:
# normalise fund holding percentages
df['fund_mv_total'] = df.groupby(['i Position Effective Date', 'Entity ID'])['Sum of Market Value Income'].transform('sum')
df['% of Total Market Value'] = df['Sum of Market Value Income'] / df['fund_mv_total'] * 100

df['fund_ce_total'] = df.groupby(['i Position Effective Date', 'Entity ID'])['Current Exposure'].transform('sum')
df['Current Exposure %'] = df['Current Exposure'] / df['fund_ce_total'] * 100

# # check percentages
# df.groupby(['i Position Effective Date', 'Entity ID'])['% of Total Market Value'].sum()
# df.groupby(['i Position Effective Date', 'Entity ID'])['Current Exposure %'].sum()

In [29]:
# remove lagging 00:00:00 from the date column
df['i Position Effective Date'] = df['i Position Effective Date'].dt.date

In [30]:
# list the holdings dates
dates = df['i Position Effective Date'].unique()

In [31]:
# get leverage values

start_time = time.time()
print(f"\nCalculating {len(files) * len(funds)} exposures \
for {len(funds)} funds over {len(files)} periods ...")

# define exclusions, being cash and cash equivalents and synthetic cash, per AIFMD
excl = ['CASH', 'MONEY MARKET', 'UNKNOWN', 'SYTH']

vals = []
for dt in tqdm(dates):

    # loop through each fund for that date
    for fund in funds:
        #dataframe the fund holdings
        df_x = df[(df['i Position Effective Date'] == dt) & (df['Entity ID'] == fund)]

        # calculate market value and effective exposure
        mv = df_x.loc[df_x["Entity ID"] == fund, "Sum of Market Value Income"].sum()
        ce = df_x.loc[df_x["Entity ID"] == fund, "Current Exposure"          ].sum()

        # sum exposure
        gross      = df_x.loc[(df_x["Entity ID"] == fund) & (~df_x["Valuation First Level"].isin(excl)), "Current Exposure"].abs().sum()
        commitment = abs(df_x.loc[(df_x["Entity ID"] == fund) & (~df_x["Valuation First Level"].isin(excl)), "Current Exposure"].sum())

        # calculate leverage
        leverage_gross      = gross / mv
        leverage_commitment = commitment / mv
     
        # populate the summary dataframe
        new_data = [dt, fund, mv, gross, commitment, leverage_gross, leverage_commitment, mv - ce]
        vals.append(new_data)

# dataframe the calculation results
column_names = ['Date', 'Fund', 'NAV', 'Exposure (Gross)', 'Exposure (Commitment)', 'Leverage (Gross)', 'Leverage (Commitment)', 'MV - CE']
summary_df = pd.DataFrame(vals)
summary_df.columns = column_names
summary_df = summary_df.sort_values(by = 'Date', ascending = False)
summary_df = summary_df.reset_index(drop = True)

print(f" {timediff(start_time, time.time())} calculating \
{len(files) * len(funds)} exposures for \
{len(funds)} funds over {len(files)} periods ...\n")


Calculating 252 exposures for 21 funds over 12 periods ...


100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [00:01<00:00,  6.75it/s]

 1.8sec calculating 252 exposures for 21 funds over 12 periods ...



In [32]:
# calculate averages and standard deviations over the entire period

start_time = time.time()
print(f"\nCalculating averages and standard deviations for the {len(funds)} funds")

# scale = np.sqrt(len(files))
scale = 1
rptDate = summary_df['Date'].max()
averages = []

for fund in funds:
    df_a = summary_df[summary_df['Fund'] == fund]
    average_gross = df_a['Leverage (Gross)'].mean()
    average_commitment = df_a['Leverage (Commitment)'].mean()
    stddev_gross = df_a['Leverage (Gross)'].std() * scale
    stddev_commitment = df_a['Leverage (Commitment)'].std() * scale
    new_data = [rptDate, fund, average_gross, stddev_gross, average_commitment, stddev_commitment]
    averages.append(new_data)

# dataframe the calculation results
col_names = [f'{len(files)} months ended {rptDate.strftime("%d%b%Y")}', 
             'Fund', 'Average Leverage (Gross)', 'Std Dev Leverage (Gross)', 
            'Average Leverage (Commitment)', 'Std Dev Leverage (Commitment)']
leverages = pd.DataFrame(averages)
leverages.columns = col_names

# leverages

print(f" {timediff(start_time, time.time())} calculating averages and standard deviations for the {len(funds)} funds\n")


Calculating averages and standard deviations for the 21 funds
 0.0sec calculating averages and standard deviations for the 21 funds



In [55]:
# Proportion (% of NAV) of illiquid assets that the fund structurally holds 
# (or is expected to hold). [Hedge Funds, Funds of Hedge Funds, 
# Physical Real Estate, Real Estate Funds, Infrastructure fund, 
# Private Equity, Private Equity Funds, Precious Metals, 
# Loans, and Non-agency ABS]

rptDate = pd.to_datetime('2025-11-30').date()

In [56]:
# check what kinds of assets are held in the funds
all_assets = df.loc[(~df["Valuation First Level"].isin(excl)), "Valuation First Level"].unique()
all_assets

array(['COLLECTIVE INVESTMENT SCHEMES', 'DERIVATIVES', 'EQUITIES',
       'FORWARDS', 'BONDS', 'PRIVATE COMPANY', 'SWAPS',
       'PREFERENCE SHARES'], dtype=object)

In [57]:
# check 'PRIVATE COMPANY'
print(df[(df['Valuation First Level'] == "PRIVATE COMPANY")]["PrimaryAssetID"].unique())

['PIMEVOA']


In [58]:
# check out the prefs held in the funds
print(df[(df['Valuation First Level'] == "PREFERENCE SHARES")]["PrimaryAssetID"].unique())

['ABSP' 'SBPP']


In [59]:
# list the 'illiquid' assets
illiquid_assets = ['PIMEVOA','SVG049', 'STP787']

In [60]:
# show the funds holding 'illiquid' assets
illiquid_holdings = df[(df['i Position Effective Date'] == rptDate) & (df['PrimaryAssetID'].isin(illiquid_assets))][['Entity ID', 'Current Exposure %']]
illiquid_holdings

,Entity ID,Current Exposure %
5455,PCGEARF,40.483367
5456,PCGEARF,12.276745
5761,PIMBAL,4.609313
6360,PABS,4.721071
6793,BPROV,4.598608
6820,UCTRFBAL,4.668229
6856,UNISABAL,4.715997


In [61]:
# If % of NAV for high-risk asset >25%, please provide the split of illiquid assets 
# [Hedge Funds, Funds of Hedge Funds, Physical Real Estate, Real Estate Funds, 
# Private Equity, Private Equity Funds, Infrastructure fund, Precious Metals, 
# Loans, and Non-agency ABS]

In [62]:
# add a new column naming each holding by code and description
df['Asset Description'] = df['Issue Description'] + \
' (' + df['PrimaryAssetID'].astype(str) + ') ' + \
round(df['Current Exposure %'],1).astype(str) + '%'
df['Asset Description']

0        BNP Paribas Margin Account - HKD (BNPMARGHKD) ...
1              China Minimum Reserve Fund (CNYMINRES) 0.1%
2        Firstrand London FD 5.17% 241125 (FSRLFD241125...
3        Firstrand London FD 5.22% 060126 (FSRLFD060126...
4        Firstrand London FD 5.24% 080126 (FSRLFD080126...
                               ...                        
16900    Synthetic Cash_EURO-BUND FUTURE Mar26 (RXH6) -...
16901    Synthetic Cash_LONG GILT FUTURE Mar26 (G H6) -...
16902    Synthetic Cash_US 10YR NOTE (CBT) Mar26 (TYH6)...
16903    Synthetic Cash_US 2YR NOTE (CBT) Mar26 (TUH6) ...
16904    Synthetic Cash_US ULTRA BOND CBT Mar26 (WNH6) ...
Name: Asset Description, Length: 16905, dtype: object

In [63]:
# make a subset of holdings excluding 'CASH' and 'UNKNOWN'
df_ex_CASH = df[~df['Valuation First Level'].isin(['CASH', 'UNKNOWN', 'FORWARDS', 'SWAPS', 'DERIVATIVES'])]

In [65]:
# get top ten holdings for each fund and for each date

# Claude prompt 2 Feb 2026: given a dataframe of serval funds and their 
# respective holdings overa several dates, how to get the 
# top ten holdings of each fund over each of the dates
top10 = (df_ex_CASH.groupby(['i Position Effective Date', 'Entity ID'])
           .apply(lambda x: x.nlargest(10, '% of Total Market Value'), include_groups=False)
           .reset_index(level=[0, 1])
           .reset_index(drop=True))
# reset_index(level=[0, 1]) pulls # the date and fund 
# groupby keys back in as columns, then
# reset_index(drop=True) cleans up the leftover numeric index

In [66]:
# create a new column concatenating the top ten holdings

# Geini prompt 2 Feb 2026: "for each date and each fund,
# create a new column combining the securityu names 
# making up the top ten holdings into a comm-separated string"
top10_summary = (top10.groupby(['i Position Effective Date', 'Entity ID'])['Asset Description']
                      .apply(lambda x: ', '.join(x))
                      .reset_index())
# top10_summary
# top10_summary.columns = ['i Position Effective Date', 'Entity ID', 'Asset Description']

In [67]:
# write the summary and leverage dataframe to Excel

start_time  = time.time()
filename    = pthL + '\\' + f'Fund Leverages {rptDate.strftime("%d%b%Y")}_b.xlsx'
with pd.ExcelWriter(filename, engine  = 'xlsxwriter') as writer:
    leverages.to_excel(        writer, index = False, sheet_name = 'averages' )
    summary_df.to_excel(       writer, index = False, sheet_name = 'exposures')
    illiquid_holdings.to_excel(writer, index = False, sheet_name = 'illiquids')
    top10_summary.to_excel(    writer, index = False, sheet_name = 'top10'    )
    df.to_excel(               writer, index = False, sheet_name = 'holdings' )
    
writer.close()

print(filename)

print(f'\n{timediff(start_time, time.time())} writing the dataframes to a file\n')

print(f"\n\n{timediff(start_time_lvrg, time.time())} total time\n")

\\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\BNP Leverage DD\Annual Review Data\Fund Leverages 30Nov2025_b.xlsx

22.9sec writing the dataframes to a file



26min 2.7sec total time



C:\Users\hilton.netta\AppData\Local\anaconda3\Lib\site-packages\xlsxwriter\workbook.py:368: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")


In [68]:
# !jupyter nbconvert --to script leverage.ipynb # convert from .ipynb to .py

[NbConvertApp] WARNING | pattern '#' matched no files
[NbConvertApp] WARNING | pattern 'convert' matched no files
[NbConvertApp] WARNING | pattern 'from' matched no files
[NbConvertApp] WARNING | pattern '.ipynb' matched no files
[NbConvertApp] WARNING | pattern 'to' matched no files
[NbConvertApp] WARNING | pattern '.py' matched no files
[NbConvertApp] Converting notebook leverage.ipynb to script
[NbConvertApp] Writing 9119 bytes to leverage.py
